#Downloads and imports (~30s)

In [1]:
import random
import time
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import yfinance as yf
import pandas as pd
from matplotlib import pyplot as plt
import math
from dateutil.relativedelta import relativedelta

#Functions (~10s)


In [2]:
def change_days(date_str, days):
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

def backtest_exit_rules(stock, date, stoploss = 0.1 ,  action = 'long', timeframe = 'month', days = 0):#function will return how much the stock price changed in the coming month with exit rules
  if timeframe == 'month':
    enddate = change_months(date, 1)
  if timeframe == 'week':
    enddate = change_days(date, 7)
  if timeframe == 'custom':
    enddate = change_days(date, days)
  print('\n',f"[------=====◼️◼️◼️◼️◼️ Backtesting {action} {stock} [{date} --> {enddate}] ◼️◼️◼️◼️◼️=====--------]")

  initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
  new_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)

  #print(order)
  pct_change = (float(new_price["Close"].iloc[0])-float(initial_price["Close"].iloc[0]))/float(initial_price["Close"].iloc[0])

  print(f' initial close: {initial_price["Close"].iloc[0]}, new close: {new_price["Close"].iloc[0]}')

  if action == 'long':
    order = yf.Ticker(stock).history(start=date, end=enddate)['Low']
    stop_loss_price = float(initial_price["Close"].iloc[0])*(1-stoploss)
    for i in order:
      if i < stop_loss_price:
        print(f"[---------🔴 Long {stock} price decreased by over {stoploss*100}%, stop loss triggered 🔴 ---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change} ---------]")

    return pct_change

  elif action =='short':
    order = yf.Ticker(stock).history(start=date, end=enddate)['High']
    stop_loss_price = float(initial_price['Close'].iloc[0])*(1+stoploss)
    for i in order:
      if i > stop_loss_price:

        print(f"[---------🔴 Short {stock} price increased by over {stoploss*100}%, stop loss triggered 🔴---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change*-1} ---------]")
    return pct_change*-1



In [3]:
def backtest_minute(stock, date, stoploss = 0.1 ,  action = 'long', timeframe = 'month', days = 0):#function will return how much the stock price changed in the coming month with exit rules
  if timeframe == 'month':
    enddate = change_months(date, 1)
  if timeframe == 'week':
    enddate = change_days(date, 7)
  if timeframe == 'custom':
    enddate = change_days(date, days)
  print('\n',f"[------=====◼️◼️◼️◼️◼️ Backtesting {action} {stock} [{date} --> {enddate}] ◼️◼️◼️◼️◼️=====--------]")





  initial_price = yf.Ticker(stock).history(start = change_days(date, -5), end = date).tail(1)
  final_price = yf.Ticker(stock).history(start = date, end = enddate).tail(1)
  initial_price = float(initial_price['Close'].iloc[0])
  final_price = float(final_price['Close'].iloc[0])




  #print(order)
  pct_change = (final_price-initial_price)/initial_price

  print(f' initial close: {initial_price}, new close: {final_price}')

  if action == 'long':
    order = yf.Ticker(stock).history(start=date, end=enddate, interval = '2m')['Low']
    stop_loss_price = initial_price*(1-stoploss)
    for i in order:
      if i < stop_loss_price:
        print(f"[---------🔴 Long {stock} price decreased by over {stoploss*100}%, stop loss triggered, loss: {(i-initial_price)/initial_price} 🔴 ---------]")
        return (i-initial_price)/initial_price

    print(f"[--------- {stock} Percentage Gain: {pct_change} ---------]")

    return pct_change

  elif action =='short':
    order = yf.Ticker(stock).history(start=date, end=enddate, interval = '2m')['High']
    stop_loss_price = initial_price*(1+stoploss)
    for i in order:
      if i > stop_loss_price:

        print(f"[---------🔴 Short {stock} price increased by over {stoploss*100}%, stop loss triggered, loss: {(initial_price-i)/initial_price} 🔴---------]")
        return (initial_price-i)/initial_price

    print(f"[--------- {stock} Percentage Gain: {pct_change*-1} ---------]")
    return pct_change*-1



  else:
    print('🟥🟥🟥🟥🟥 PLEASE ENTER ACTION (LONG/SHORT) 🟥🟥🟥🟥🟥')
  return pct_change


# MN + SL test


In [4]:
#All S&P500 stocks in 20 and 26
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'PCAR', 'PKG', 'PANW', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

482

In [5]:
date = '2026-04-01'
date_ = date
stoploss = 0.001
timeframe = 'custom'
days = 3
repititions = 47
date_increment = 1

stocks_mem = stocks[:]
index = []
changes = []

for i in range(len(stocks)):
  index.append(i)

for i in range(3):
  date = date_
  for i in range(repititions):
    try:
      print('\n'*3, f'[==========================================={date}================================================]')
      time.sleep(0.5) # To prevent yfinance from crashing and limiting fetches
      long_changes = []
      short_changes = []

      chosen_indexes = random.choices(index, k=10)

      bull_stocks = chosen_indexes[:5]
      bear_stocks = chosen_indexes[5:]

      for j in range(5):
        try:
          long_changes.append(backtest_minute(stocks[bull_stocks[j]], date, stoploss=stoploss, action = 'long', timeframe = timeframe, days=days))
        except:
          pass
        try:
          short_changes.append(backtest_minute(stocks[bear_stocks[j]], date, stoploss=stoploss, action = 'short', timeframe = timeframe, days=days))
        except:
          pass
      long_change = sum(long_changes)/len(long_changes)
      short_change = sum(short_changes)/len(short_changes)

      months_change = long_change + short_change

      changes.append(months_change)

      if timeframe == 'month':
        date = change_months(date, 1)
      else:
        date = change_days(date, date_increment)
    except:
      if timeframe == 'month':
        date = change_months(date, 1)
      else:
        date = change_days(date, date_increment)




 [===========================================2026-04-01================================================]

 [------=====◼️◼️◼️◼️◼️ Backtesting long SHW [2026-04-01 --> 2026-04-04] ◼️◼️◼️◼️◼️=====--------]
 initial close: 320.54998779296875, new close: 318.0
[---------🔴 Long SHW price decreased by over 0.1%, stop loss triggered, loss: -0.025237861943440944 🔴 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting short IDXX [2026-04-01 --> 2026-04-04] ◼️◼️◼️◼️◼️=====--------]
 initial close: 561.8900146484375, new close: 569.5499877929688
[---------🔴 Short IDXX price increased by over 0.1%, stop loss triggered, loss: -0.0044759909997886165 🔴---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting long CMCSA [2026-04-01 --> 2026-04-04] ◼️◼️◼️◼️◼️=====--------]
 initial close: 28.3799991607666, new close: 27.93000030517578
[---------🔴 Long CMCSA price decreased by over 0.1%, stop loss triggered, loss: -0.010923166846704344 🔴 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting short NI [2026-04-01 --> 2026-0

ERROR:yfinance:$CPB: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-06)



 [------=====◼️◼️◼️◼️◼️ Backtesting short AJG [2026-04-03 --> 2026-04-06] ◼️◼️◼️◼️◼️=====--------]


ERROR:yfinance:$AJG: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-06)



 [------=====◼️◼️◼️◼️◼️ Backtesting long CAH [2026-04-03 --> 2026-04-06] ◼️◼️◼️◼️◼️=====--------]


...

ERROR:yfinance:$ENPH: possibly delisted; no price data found  (1d 2026-05-17 -> 2026-05-20) (Yahoo error = "Data doesn't exist for startDate = 1778990400, endDate = 1779249600")



 [------=====◼️◼️◼️◼️◼️ Backtesting long NKE [2026-05-17 --> 2026-05-20] ◼️◼️◼️◼️◼️=====--------]


ERROR:yfinance:$NKE: possibly delisted; no price data found  (1d 2026-05-17 -> 2026-05-20) (Yahoo error = "Data doesn't exist for startDate = 1778990400, endDate = 1779249600")



 [------=====◼️◼️◼️◼️◼️ Backtesting short SLB [2026-05-17 --> 2026-05-20] ◼️◼️◼️◼️◼️=====--------]


ERROR:yfinance:$SLB: possibly delisted; no price data found  (1d 2026-05-17 -> 2026-05-20) (Yahoo error = "Data doesn't exist for startDate = 1778990400, endDate = 1779249600")


In [6]:
changes

[-0.011440859075146765,
 -0.0045446103632249945,
 -0.007736380052895967,
 -0.0009761828251292079,
 -0.007700068509018335,
 -0.023738878352500003,
 -0.001225709298505448,
 -0.001056215541890465,
 0.00596656729781517,
 -0.0076674351027267214,
 -0.007843879800095776,
 -0.012978139381565498,
 -0.0037863187054956553,
 -0.024468367342253883,
 0.002156144463610082,
 0.03977787151917083,
 -0.00961011043924378,
 -0.008555554136444442,
 -0.01712059596723961,
 0.008114921447420433,
 -0.007186874204186157,
 -0.02120025499142772,
 -0.014422316512279291,
 -0.013209384066988036,
 0.004866373130875531,
 -0.01664771375957539,
 -0.024789374160077888,
 -0.011297721609288939,
 -0.010648457149445172,
 0.0012186914738302954,
 -0.008822475754002069,
 -0.005053696621831913,
 -0.019019292124833484,
 0.0326094527897316,
 0.002771469568081327,
 -0.007395900635699236,
 0.0011139710636527108,
 -0.01238909090368156,
 0.010355191935923101,
 -0.008516300887233461,
 -0.015423449561698787,
 -0.01192370018832691,
 0.0,


In [7]:
p = 0
wins = []
for i in changes:
  if i > 0:
    p+=1
    wins.append(i)
print(len(changes), p, p/len(changes))

129 31 0.24031007751937986


In [8]:
sum(wins)/len(wins), len(wins)

(0.008823911544182862, 31)

In [9]:
import statistics

print(f'''
Changes Summary:
----------------
Count          : {len(changes)}
Mean           : {statistics.mean(changes):.6f}
Median         : {statistics.median(changes):.6f}
Min            : {min(changes):.6f}
Max            : {max(changes):.6f}
Standard Dev   : {statistics.stdev(changes):.6f}
Variance       : {statistics.variance(changes):.6f}
25th Percentile: {sorted(changes)[int(0.25 * len(changes))]:.6f}
50th Percentile: {statistics.median(changes):.6f}
75th Percentile: {sorted(changes)[int(0.75 * len(changes))]:.6f}
----------------
Sum            : {sum(changes):.6f}
''')


Changes Summary:
----------------
Count          : 129
Mean           : -0.006747
Median         : -0.007736
Min            : -0.073769
Max            : 0.039778
Standard Dev   : 0.012947
Variance       : 0.000168
25th Percentile: -0.012978
50th Percentile: -0.007736
75th Percentile: 0.000000
----------------
Sum            : -0.870384



In [10]:
changes_plus_one = []
for i in changes:
  changes_plus_one.append(i+1)

k=1
for i in changes_plus_one:
  k = k*i

print('Compounded Gain:',(k-1)*100, '%')

Compounded Gain: -58.698296252521565 %


In [ ]:
len(changes_plus_one)

1191

In [ ]:
p_dict = {}

for i in range(0, len(changes_plus_one), 3):
    # Calculate the index (1, 2, 3...)
    n = (i // 3) + 1
    # Slice the list and store in the dictionary
    p_dict[f'P{n}'] = changes_plus_one[i:i+3]

# Access them like this:
print(p_dict['P1'])
print(p_dict['P2'])

[1.0012659487440907, 1.0292611935246365, 1.0044499939464033]
[0.9992538948622999, 0.998, 1.001409966222417]


In [ ]:
print('2020-2025 change:')
for i in p_dict.values():
  k=1
  for j in i:
    k = k*j
  print(k)

2020-2025 change:
1.035150189826508
0.9986614834834711
0.9953158938211534
1.023561400776646
1.0052639351207089
0.9947165158009587
1.00684264716172
1.001271929325438
1.015267821876827
1.0318228697198042
0.994011992
1.0097145875775029
1.0251741345541585
0.9957682386360037
0.994011992
1.0077719928147653
0.9989603937981072
0.9990481195725707
1.0098509042722659
1.0169926948370795
1.0272512616183633
1.0192602788112475
0.9989772527082074
1.0083211555925866
1.0225915941433297
1.05698111619666
1.0292769324875441
1.0319197516987584
0.9973577539095385
1.05536432958896
1.0057439712015575
1.0009340713227262
1.0282228276722174
0.9996311195582647
1.0403175823045774
1.003473700182561
0.9951863237831773
1.0330205400747174
1.0021805735800338
1.0192216830621592
1.018823635935968
0.994011992
1.0299376780964238
1.0545121645130182
1.0307721855588734
1.0328693699427116
1.0097507915063149
1.023447426908414
1.0260643234167632
1.0072244285325338
1.0012115788726548
1.00273331313307
0.994011992
1.0223474379246078

In [ ]:
for i in range(1): # this line is to make this code collapsable
  date = '2015-01-01' ########DONT USE THIS CODE, IT GETS STUCK
  stoploss = 0.001
  timeframe = 'custom'
  days = 3
  reps = 1216

  stocks_mem = stocks[:]
  index = []
  changes = []

  for i in range(len(stocks)):
    index.append(i)

  for i in range(reps):
    print('\n'*3, f'[==========================================={date}================================================]')
    time.sleep(0.5) # To prevent yfinance from crashing and limiting fetches
    long_changes = []
    short_changes = []

    chosen_indexes = random.choices(index, k=30)

    bull_stocks = chosen_indexes[:5]
    bear_stocks = chosen_indexes[5:10]
    bull_backup_stocks = chosen_indexes[10:20]
    bear_backup_stocks = chosen_indexes[20:30]

    lk = -1
    sk = -1

    for j in range(5):
      long_running = True
      short_running = True

      try:
        long_changes.append(backtest_exit_rules(stocks[bull_stocks[j]], date, stoploss=stoploss, action = 'long', timeframe = timeframe, days=days))
      except:
        while long_running:
          len1 = len(long_changes)
          lk+=1

          try:
            long_changes.append(backtest_exit_rules(stocks[bull_backup_stocks[lk]], date, stoploss=stoploss, action = 'long', timeframe = timeframe, days=days))
          except:
            pass

          len2 = len(long_changes)
          if len1 != len2:
            long_running = False
            break
        pass

      try:
        short_changes.append(backtest_exit_rules(stocks[bear_stocks[j]], date, stoploss=stoploss, action = 'short', timeframe = timeframe, days=days))
      except:
        sk=-1
        while short_running:
          len1 = len(short_changes)
          sk+=1

          try:
            short_changes.append(backtest_exit_rules(stocks[bear_backup_stocks[sk]], date, stoploss=stoploss, action = 'short', timeframe = timeframe, days=days))
          except:
            pass

          len2 = len(short_changes)
          if len1 != len2:
            short_running = False
            break
        pass


    long_change = sum(long_changes)/len(long_changes)
    short_change = sum(short_changes)/len(short_changes)

    months_change = long_change + short_change

    changes.append(months_change)

    if timeframe == 'month':
      date = change_months(date, 1)
    elif timeframe == 'custom':
      date = change_days(date, days)

#scrap


In [ ]:
error = []
for stock in stocks:
  ticker = yf.Ticker(stock).history(start = '2026-02-18', end='2026-04-18')
  if ticker.empty:
    error.append(stock)

for i in error:
  stocks.remove(i)

print(stocks)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DAY"}}}
ERROR:yfinance:$DAY: possibly delisted; no timezone found


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DECK', 'DE', 'DE

In [ ]:
error

['DAY', 'MMC']

In [ ]:
import tpot
from tpot import TPOTClassifier

In [ ]:
stock = "aapl"
date = "2026-01-01"
X,y = data(stock, change_months(date, -72), date,shuffle = False)

X_test = X.iloc[-200:]
y_test = y.iloc[-200:]

X = X.iloc[:-222]
y = y.iloc[:-222]

X = pd.concat([X,y], axis=1)
X_train = X.sample(frac=1, random_state=1)

y_train = X_train.pop("label")

"Train: ", X_train.shape, y_train.shape, "Test:", X_test.shape, y_train.shape

('Train: ', (1197, 289), (1197,), 'Test:', (200, 289), (1197,))

In [ ]:
X_train.shape

(1197, 289)

In [ ]:
TPOT = tpot.TPOTClassifier(
    #scoring = 'accuracy',
    generations = 10, #how many interations per algorithm is tested
    population_size = 20,
    verbose=1,
    cv = 5,
    max_time_mins = 60
    )
TPOT.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:35749
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:34769'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39433 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39433
INFO:distributed.core:Starting established connection to tcp://127.0.0.1

TPOTClassifier(cv=5,
               search_space=<tpot.search_spaces.pipelines.sequential.SequentialPipeline object at 0x78d414223f50>,
               verbose=1)

In [ ]:
score = TPOT.fitted_pipeline_.score(X_test, y_test)
print(f"Test Score: {score}")
# Print the entire pipeline structure
print(TPOT.fitted_pipeline_)

# To specifically see the hyperparameters of the final model:
best_step_name = list(TPOT.fitted_pipeline_.named_steps.keys())
print(f"Model: {best_step_name}")
print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_step_name].get_params()}")


Test Score: 0.0
Pipeline(steps=[('maxabsscaler', MaxAbsScaler()),
                ('selectfwe', SelectFwe(alpha=0.0003078185684)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('featureunion-2',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('baggingclassifier',
                 BaggingClassifier(bootstrap=np.True_,
                                   bootstrap_features=np.False_,
                                   max_features=0.6524941957496,
                                   max

In [ ]:
BG = BaggingClassifier(
    bootstrap=np.True_,
    bootstrap_features=np.False_,
    estimator=None,
    max_features=0.6524941957496,
    max_samples=0.8359000837192,
    n_estimators=100,
    n_jobs=1,
    oob_score=np.False_,
    random_state=None,
    verbose=0,
    warm_start=False
)


In [ ]:
BG.fit(X_train, y_train)

BaggingClassifier(bootstrap=np.True_, bootstrap_features=np.False_,
                  max_features=0.6524941957496, max_samples=0.8359000837192,
                  n_estimators=100, n_jobs=1, oob_score=np.False_)

In [ ]:
BG.score(X_test, y_test)

0.17

In [ ]:
vote.fit(X_train, y_train)

VotingClassifier(estimators=[('ADARF',
                              AdaBoostClassifier(estimator=RandomForestClassifier(random_state=1),
                                                 random_state=1)),
                             ('RF', RandomForestClassifier(random_state=1)),
                             ('LGBM',
                              LGBMClassifier(random_state=1, verbose=-1)),
                             ('ET', ExtraTreesClassifier(random_state=1))],
                 voting='soft')

In [ ]:
best_step_name = list(TPOT.fitted_pipeline_.named_steps.keys())
print(best_step_name)

['maxabsscaler', 'selectfwe', 'featureunion-1', 'featureunion-2', 'baggingclassifier']


In [ ]:
print(f"Hyperparameters: {TPOT.fitted_pipeline_.named_steps[best_step_name[2]].get_params()}")


Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}


In [ ]:
#model name = BG

#Hyperparameters: Maxbsscaler {copy=True}
#Selectfwe {'alpha': 0.0003078185684, 'score_func': <function f_classif at 0x78d44cd9c4a0>}
#featureunion-1 Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}
#featureunion-2 Hyperparameters: {'n_jobs': None, 'transformer_list': [('skiptransformer', SkipTransformer()), ('passthrough', Passthrough())], 'transformer_weights': None, 'verbose': False, 'verbose_feature_names_out': True, 'skiptransformer': SkipTransformer(), 'passthrough': Passthrough()}


In [ ]:
import numpy as np
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_selection import SelectFwe, f_classif
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

# Assume X_train, y_train, X_test, y_test are already defined

# Step 1: Apply MaxAbsScaler with copy=True
scaler = MaxAbsScaler(copy=True)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: SelectFwe with given parameters
select_fwe = SelectFwe(alpha=0.0003078185684, score_func=f_classif)
X_train_selected = select_fwe.fit_transform(X_train_scaled, y_train)
X_test_selected = select_fwe.transform(X_test_scaled)

# Step 3: Create FeatureUnion-1
feature_union_1 = FeatureUnion(
    transformer_list=[
        ('original', 'passthrough'),  # Keeps one copy of original data
        ('scaled', StandardScaler())  # Adds a scaled copy of the data
    ],
    n_jobs=None,
    transformer_weights=None,
    verbose=False,
    verbose_feature_names_out=True
)

# Fit and transform train data, transform test data for FeatureUnion-1
X_train_fu1 = feature_union_1.fit_transform(X_train_selected, y_train)
X_test_fu1 = feature_union_1.transform(X_test_selected)

# Step 4: Create FeatureUnion-2 (same settings)
feature_union_2 = FeatureUnion(
    transformer_list=[
        ('original', 'passthrough'),  # Keeps one copy of original data
        ('scaled', StandardScaler())  # Adds a scaled copy of the data
    ],
    n_jobs=None,
    transformer_weights=None,
    verbose=False,
    verbose_feature_names_out=True
)

X_train_processed = feature_union_2.fit_transform(X_train_fu1, y_train)
X_test_processed = feature_union_2.transform(X_test_fu1)



/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [5] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [ ]:
BG.fit(X_train_processed, y_train)

BaggingClassifier(bootstrap=np.True_, bootstrap_features=np.False_,
                  max_features=0.6524941957496, max_samples=0.8359000837192,
                  n_estimators=100, n_jobs=1, oob_score=np.False_)

In [ ]:
ypred = BG.predict(X_test_processed)
cm = confusion_matrix(y_test, ypred)
cf = classification_report(y_test, ypred)

print(cm,"\n",cf)

[[ 0  0  0  0  0  0]
 [ 0  0  3  1  0  0]
 [ 1  0 38 13  0  6]
 [ 6  0 70 30  0  3]
 [ 0  0  5  1  0  0]
 [ 0  0 17  6  0  0]] 
               precision    recall  f1-score   support

          -3       0.00      0.00      0.00         0
          -2       0.00      0.00      0.00         4
          -1       0.29      0.66      0.40        58
           1       0.59      0.28      0.38       109
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00        23

    accuracy                           0.34       200
   macro avg       0.15      0.16      0.13       200
weighted avg       0.40      0.34      0.32       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_